In [5]:
import pandas as pd 

data = pd.read_csv('new-data.csv')

data.head() 

,timestamp,open,high,low,close,volume
0,2025-04-25 20:00:00,550.640,550.64,550.64,550.6400,1571349
1,2025-04-25 19:45:00,551.100,551.25,551.06,551.2293,16162
2,2025-04-25 19:30:00,551.125,551.17,550.70,551.1200,7317
3,2025-04-25 19:15:00,551.080,551.15,550.99,551.1090,8679
4,2025-04-25 19:00:00,551.140,551.15,550.85,551.0600,7862


In [6]:
# convert the timestamp to date time and set as index 
data['timestamp'] = pd.to_datetime(data['timestamp'])
data.set_index('timestamp', inplace=True)

# calculate price change and movement % 
data['price_change'] = data['close'] - data['open']

data['price_change_pct'] = (data['price_change'] / data['open']) * 100 


# 5MA 

data['5ma'] = data['close'].rolling(window=5).mean()

# momentum (price movement over last 5 periodds)

data['momentum'] = data['price_change_pct'].rolling(window=5).mean()

# volume averagte over 5 periods
data['volume_avg'] = data['volume'].rolling(window=5).mean()

# drop rows in nan from rolling calc
data.dropna(inplace=True)

data.head()

,open,high,low,close,volume,price_change,price_change_pct,5ma,momentum,volume_avg
timestamp,,,,,,,,,,
2025-04-25 19:00:00,551.14,551.1500,550.850,551.0600,7862,-0.0800,-0.014515,551.03166,0.002660,322273.8
2025-04-25 18:45:00,551.00,551.1500,550.940,551.1000,12357,0.1000,0.018149,551.12366,0.006290,10475.4
2025-04-25 18:30:00,550.64,551.0487,550.640,551.0472,1601428,0.4072,0.073950,551.08724,0.016388,327528.6
2025-04-25 18:15:00,550.63,550.8300,550.570,550.8000,11395,0.1700,0.030874,551.02324,0.022744,328344.2
2025-04-25 18:00:00,550.68,550.7000,550.565,550.6200,41155,-0.0600,-0.010896,550.92544,0.019512,334839.4


In [8]:
# targete variable based on next period close 
#1 -> bullish -1 bearish 
data['future_close'] = data['close'].shift(-1) # the next periods close
data['target'] = (data['future_close']- data['close']).apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))


# drop last row since its target is NAN cause of shift 
data.dropna(subset=['target'], inplace=True)

features  = [ 
    'price_change',
    'price_change_pct', 
    '5ma', 
    'momentum', 
    'volume_avg'
]

X = data[features]
y = data['target']

# split data into training and testing 
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)


X_train.shape, X_test.shape, y_train.shape, y_test.shape


((1088, 5), (273, 5), (1088,), (273,))

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report 

model = RandomForestClassifier(n_estimators=100, random_state=0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

report = classification_report(y_test, y_pred, digits=4)

report

/Users/jimmychavada/Documents/Snake/Mobile-ddev/DayTradingSupportTool/server/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/jimmychavada/Documents/Snake/Mobile-ddev/DayTradingSupportTool/server/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/jimmychavada/Documents/Snake/Mobile-ddev/DayTradingSupportTool/server/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0

'              precision    recall  f1-score   support\n\n          -1     0.9667    0.9355    0.9508       124\n           0     0.0000    0.0000    0.0000         1\n           1     0.9412    0.9730    0.9568       148\n\n    accuracy                         0.9524       273\n   macro avg     0.6359    0.6362    0.6359       273\nweighted avg     0.9493    0.9524    0.9506       273\n'

In [10]:
import joblib
joblib.dump(model, 'market_bias_model.pkl')



['market_bias_model.pkl']